# **Bayesian Hyperparameter Optimization with Nested Cross-Validation**

This notebook performs nested Bayesian hyperparameter optimization with cross-validation to develop and compare multiple regression models for predicting pIC50 values using the reduced feature (RFECV) descriptor set.

The goal is to:

- Identify optimal hyperparameters for each algorithm

- Obtain unbiased model performance estimates using nested CV

- Evaluate final model performance on an independent test set

- Compare algorithm robustness across targets


**Target Proteins**

The workflow supports the following Alzheimer's disease–relevant targets:

- 5-HT6

- AChE

- BACE1

- BuChE

- ESR1

- GSK-3β

- MAO-B


**Models Evaluated**

- Random Forest

- XGBoost

- Gradient Boosting

- Support Vector Regression (SVR)

- K-Nearest Neighbors (KNN)

- Custom Single-Layer MLP

The custom MLP implementation enables integer neuron search within Bayesian optimization.

**Model Validation Strategy**

**Outer Loop:**

- 5-fold K-Fold cross-validation

**Inner Loop:**

- 3-fold cross-validation

- Bayesian optimization via BayesSearchCV

- 25 optimization iterations per fold

- Optimization metric: R²


Scaling is applied where required (SVR, KNN, MLP).

**Evaluation Metrics**

For each model, the following are reported:

- Mean Nested CV R²

- Mean Nested CV RMSE

- Independent Test R²

- Independent Test RMSE

- Best hyperparameters

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
!pip install scikit-optimize


In [ ]:
import pandas as pd
import numpy as np
import os
from math import sqrt
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score, mean_squared_error
from skopt import BayesSearchCV
from skopt.space import Real, Integer, Categorical
from sklearn.base import BaseEstimator, RegressorMixin
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold



In [ ]:
# === Paths to datasets ===

reu_path = "/content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/data/descriptors/reduced_features_reu_2"
meta_cols = ['molecule_chembl_id', 'canonical_smiles', 'activity_class', 'pIC50']

In [ ]:
targets = ["5-HT6","ache", "bace1", "buche","esr1","gsk-3beta", "mao-b"]

target=targets[2]

In [ ]:
df_train = pd.read_csv(f"{reu_path}/{target}_train_reu.csv").dropna(how="any")
df_test = pd.read_csv(f"{reu_path}/{target}_test_reu.csv").dropna(how="any")

In [ ]:
X_train = df_train.drop(columns=meta_cols)
y_train = df_train["pIC50"]
X_test = df_test.drop(columns=meta_cols)
y_test = df_test["pIC50"]

In [ ]:
# === Custom MLP with Integer Neuron Search ===
class SingleLayerMLP(BaseEstimator, RegressorMixin):
    def __init__(self, n_neurons=100, alpha=0.001, max_iter=1000, random_state=42):
        self.n_neurons = n_neurons
        self.alpha = alpha
        self.max_iter = max_iter
        self.random_state = random_state

    def fit(self, X, y):
        self.model_ = MLPRegressor(
            hidden_layer_sizes=(self.n_neurons,),
            alpha=self.alpha,
            max_iter=self.max_iter,
            random_state=self.random_state
        )
        self.model_.fit(X, y)
        return self

    def predict(self, X):
        return self.model_.predict(X)

In [ ]:
# === Define Models and Search Spaces ===
models = {
    "RandomForest": (
        RandomForestRegressor(random_state=42),
        {
            "model__n_estimators": Integer(50, 500),
            "model__max_depth": Integer(5, 50),
            "model__min_samples_split": Integer(2, 20)
        }
    ),
    "XGBoost": (
        XGBRegressor(random_state=42, objective="reg:squarederror", verbosity=0),
        {
            "model__n_estimators": Integer(50, 300),
            "model__max_depth": Integer(3, 10),
            "model__learning_rate": Real(0.01, 0.3, prior="log-uniform"),
            "model__subsample": Real(0.5, 1.0),
            "model__colsample_bytree": Real(0.5, 1.0)
        }
    ),

    "GradientBoosting": (
        GradientBoostingRegressor(random_state=42),
        {
            "model__n_estimators": Integer(50, 300),
            "model__learning_rate": Real(0.01, 0.3, prior="log-uniform"),
            "model__max_depth": Integer(3, 10)
        }
    ),
     "SVR": (
        SVR(),
        {
            "model__C": Real(0.1, 10.0, prior="log-uniform"),
            "model__kernel": Categorical(["linear", "rbf"])
        }
    ),
    "KNeighborsRegressor": (
        KNeighborsRegressor(),
        {
            "model__n_neighbors": Integer(3, 15)
        }
    ),
    "SingleLayerMLP": (
        SingleLayerMLP(),
        {
            "model__n_neurons": Integer(50, 200),
            "model__alpha": Real(1e-5, 1e-1, prior="log-uniform")
        }
    )

}

In [ ]:

# === Optimization Loop ===


outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)

nested_results = []

for name, (model, search_space) in models.items():
    print(f"\n Nested CV for {name}...")

    r2_outer_scores = []
    rmse_outer_scores = []

    for train_idx, val_idx in outer_cv.split(X_train):
        X_train_outer, X_val_outer = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_train_outer, y_val_outer = y_train.iloc[train_idx], y_train.iloc[val_idx]

        # Scaling pipeline if needed
        if name in ["SVR", "KNeighborsRegressor", "SingleLayerMLP"]:
            pipeline = Pipeline([
                ("scaler", StandardScaler()),
                ("model", model)
            ])
        else:
            pipeline = Pipeline([
                ("model", model)
            ])

        # Inner CV: BayesSearchCV
        opt = BayesSearchCV(
            pipeline,
            search_spaces=search_space,
            n_iter=25,
            cv=3,  # Inner CV
            scoring="r2",
            n_jobs=-1,
            random_state=42
        )

        # Fit with inner CV on outer training fold
        opt.fit(X_train_outer, y_train_outer)

        # Evaluate on outer validation fold
        y_pred_outer = opt.predict(X_val_outer)
        r2_outer = r2_score(y_val_outer, y_pred_outer)
        rmse_outer = sqrt(mean_squared_error(y_val_outer, y_pred_outer))

        r2_outer_scores.append(r2_outer)
        rmse_outer_scores.append(rmse_outer)

    # Final model trained on full training set
    final_opt = BayesSearchCV(
        pipeline,
        search_spaces=search_space,
        n_iter=25,
        cv=3,
        scoring="r2",
        n_jobs=-1,
        random_state=42
    )
    final_opt.fit(X_train, y_train)
    best_model = final_opt.best_estimator_

    # Evaluate on test set
    y_pred_test = best_model.predict(X_test)
    test_r2 = r2_score(y_test, y_pred_test)
    test_rmse = sqrt(mean_squared_error(y_test, y_pred_test))

    nested_results.append({
        "Target": {target},
        "Model": name,
        "Best Params": final_opt.best_params_,
        "Nested CV R2": round(np.mean(r2_outer_scores), 4),
        "Nested CV RMSE": round(np.mean(rmse_outer_scores), 4),
        "Test R2": round(test_r2, 4),
        "Test RMSE": round(test_rmse, 4)
    })






In [ ]:
# === Save and Display ===
df_results = pd.DataFrame(nested_results)


display(df_results)

In [ ]:
df_results.head()

In [ ]:
df_results.iloc[:, 2]

In [ ]:
# === Save results ===

results_out = f"/content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/Hyperparameter-optimization/{target}_optimized_hyperparameters_reu_nested_CV_baysean_opt_4.csv"
df_results.to_csv(results_out, index=False)

print("\n Saved results to:", results_out)

In [ ]:
print("test")